In [ ]:
import numpy as np
import znnl as nl

import pandas as pd

from neural_tangents import stax
import optax

import matplotlib.pyplot as plt

import h5py as hf
from scipy.stats import pearsonr

## Download the data

In [ ]:
class AbaloneData(nl.data.DataGenerator):
    """
    Generator for the Abalone data-set.
    
    Notes
    -----
    
    URL: http://archive.ics.uci.edu/static/public/1/abalone.zip
    """
    
    def __init__(self, data_file: str = "abalone.data"):
        """
        Build the data-set.
        """
        self.data_file = data_file
        self.columns = [
            "Sex",
            "Length", 
            "Diameter", 
            "Height", 
            "Whole weight", 
            "Shucked weight", 
            "Viscera weight", 
            "Shell weight", 
            "Rings"
        ]
        
        # Collect the processed data
        processed_data = self._process_raw_data()
        
        # Create the data-sets
        train_ds = processed_data.sample(frac=0.8, random_state=0)
        train_labels = train_ds.pop("Rings")
        
        test_ds = processed_data.drop(train_ds.index)
        test_labels = test_ds.pop("Rings")
        
        self.train_ds = {
            "inputs": train_ds.to_numpy(), 
            "targets": train_labels.to_numpy().reshape(-1, 1)
        }
        self.test_ds = {
            "inputs": test_ds.to_numpy(), 
            "targets": test_labels.to_numpy().reshape(-1, 1)
        }
        
        self.data_pool = self.train_ds["inputs"]
        
        
    def _process_raw_data(self):
        """
        Process the raw data
        """
        # Process the raw data.
        raw_data = pd.read_csv(
            self.data_file, names=self.columns, na_values='?', comment='#',
                          sep=',', skipinitialspace=True
        )
        raw_data.dropna()
        
        # encode the sex data
        raw_data = pd.get_dummies(raw_data, columns=['Sex'], prefix='', prefix_sep='')
        # Normalize
        raw_data = (raw_data - raw_data.mean()) / raw_data.std()

        return raw_data
        
        

In [ ]:
generator = AbaloneData()

## Create the model

In [ ]:
def create_model(generator = generator, alpha: float = 1.0, ds_size: int = 100, subset_size: int = 1.0):
    network = stax.serial(
        stax.Dense(128),
        stax.Relu(),
        stax.Dense(128),
        stax.Relu(),
        stax.Dense(1),
    )
    
    indices = np.random.randint(
        0, generator.train_ds["inputs"].shape[0] - 1, size=ds_size
    )
    
    train_ds = {
        "inputs": np.take(generator.train_ds["inputs"], indices, axis=0),
        "targets": np.take(generator.train_ds["targets"], indices, axis=0)
    }
    
    optimizer = nl.optimizers.TraceOptimizer(
        scale_factor=alpha, subset=subset_size
    )
    model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                input_shape=(1, 10),
                batch_size=5
        )
    
    return model, train_ds
    

# Trace Convergence

Here we want to assess whether the NTK trace converges for some data-set size. 
We do this by training models using larger data-sets and recording the trace for each one.
In all cases, we take 100 epochs and use the trace optimizer.

In [ ]:
ds_sizes = [10, 100, 200, 300, 500, 1000]
    
model, train_ds = create_model(ds_size=500, subset_size=0.1)

train_recorder = nl.training_recording.JaxRecorder(
    name=f"trace-convergence/sub_train_recorder",
    loss=True,
    trace=True,
    eigenvalues=True,
    update_rate=1
)

test_recorder = nl.training_recording.JaxRecorder(
    name=f"trace-convergence/sub_test_recorder",
    loss=True,
    update_rate=1
)

subset_recorders = [
    nl.training_recording.JaxRecorder(
        name=f"trace-convergence/sub_train_recorder_{item}",
        loss=True,
        trace=True,
        eigenvalues=True,
        subset=item,
        update_rate=1
    )
    for item in ds_sizes
]

train_recorder.instantiate_recorder(
data_set=train_ds
)
test_recorder.instantiate_recorder(
    data_set=generator.test_ds
)

for item in subset_recorders:
    item.instantiate_recorder(train_ds)

training_strategy = nl.training_strategies.SimpleTraining(
    model=model, 
    loss_fn=nl.loss_functions.LPNormLoss(order=2),
    recorders=[train_recorder, test_recorder] + subset_recorders
)

_ = training_strategy.train_model(
    train_ds=train_ds, test_ds=generator.test_ds, epochs=30, batch_size=256
)
train_recorder.dump_records()
test_recorder.dump_records()
for item in subset_recorders:
    item.dump_records()

In [ ]:
def load_data(file):
    with hf.File(file, "r") as db:
        data = db["trace"][:]
        
    return data

In [ ]:
ds_sizes = [10, 100, 200, 300, 500, 1000]
data = {}

data[2000] = load_data("trace-convergence/sub_train_recorder.h5")

for item in ds_sizes:
    data[item] = load_data(f"trace-convergence/sub_train_recorder_{item}.h5") 

In [ ]:
y = []
yerr = []
x = list(data)

for item in data:
    difference = abs(data[2000][0] / data[2000] - data[item][0] / data[item])
    y.append(np.mean(difference))
    yerr.append(np.std(difference))

In [ ]:
plt.errorbar(x, y, yerr=yerr, marker="o", linestyle="none", mfc="none", capsize=5)
plt.xlabel(r"$\mathbb{S}$")
plt.ylabel(r"$\mathbb{E}_{t}\left|\eta^{2000}_{t} - \eta^{\mathbb{S}}_{t}\right|$")
plt.savefig("abalone-trace-convergence.pdf")
plt.show()